## 1. YELP - Exploratory Data Analysis (EDA) 

Dataset

https://www.kaggle.com/datasets/yelp-dataset/yelp-dataset?resource=download


EDA Process

1. Understand the business problem
2. Understand the data model
3. Data cleaning and wrangling
4. Business Intelligence and Reporting


Author [Ashish Pujari](apujari@uchicago.edu)


In [1]:
import time
import json
from numpy import array, mean
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from pyspark.sql.functions import year
from pyspark.sql.functions import from_json, col, udf, schema_of_json, size, explode, collect_set, map_keys
from pyspark.sql.functions import isnan, when, count, col,lower, split, trim, concat_ws
from pyspark.sql.types import BooleanType, MapType, StructType, StructField, StringType

from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

### Spark Application

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("EDA-Yelp").getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 16:26:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/16 16:26:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
#review configuration settings
spark.sparkContext.getConf().getAll()

[('spark.driver.host', 'jkanellopoulos-1.c.adsp-31011-ip07-f26.internal'),
 ('spark.rdd.compress', 'True'),
 ('spark.app.name', 'EDA-Yelp'),
 ('spark.hadoop.fs.s3a.vectored.read.min.seek.size', '128K'),
 ('spark.executor.extraJavaOptions',
  '-Djava.net.preferIPv6Addresses=false -XX:+IgnoreUnrecognizedVMOptions --add-modules=jdk.incubator.vector --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang.reflect=ALL-UNNAMED --add-opens=java.base/java.io=ALL-UNNAMED --add-opens=java.base/java.net=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED --add-opens=java.base/java.util.concurrent=ALL-UNNAMED --add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED --add-opens=java.base/jdk.internal.ref=ALL-UNNAMED --add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/sun.nio.cs=ALL-UNNAMED --add-opens=java.base/sun.security.action=ALL-UNNAMED --add-opens=java.base

### Read Data

In [ ]:
# Download Yelp JSON files from GCS
# Workaround only; not recommended for large datasets

from google.cloud import storage
import random
import os

client = storage.Client()
bucket = client.bucket("adsp-31011-ip07-f26-bucket")

# Download approximately 20% of a JSON dataset
def download_sample(blob_name, local_path, sample_fraction=0.10):
    blob = bucket.blob(blob_name)

    random.seed(42)

    with blob.open("r") as source, open(local_path, "w") as destination:
        for line in source:
            if random.random() < sample_fraction:
                destination.write(line)

    print(f"Downloaded approximately {sample_fraction:.0%} of {blob_name}")


# Download 20% of user data
download_sample(
    "yelp/yelp_academic_dataset_user.json",
    "/tmp/yelp_academic_dataset_user.json"
)

# Download 20% of review data
download_sample(
    "yelp/yelp_academic_dataset_review.json",
    "/tmp/yelp_academic_dataset_review.json"
)

# Download the remaining datasets in full
for filename in [
    "yelp/yelp_academic_dataset_business.json",
    "yelp/yelp_academic_dataset_checkin.json",
    "yelp/yelp_academic_dataset_tip.json"
]:
    blob = bucket.blob(filename)

    # Remove the GCS folder prefix for the local filename
    local_path = f"/tmp/{os.path.basename(filename)}"

    blob.download_to_filename(local_path)

    print(f"Downloaded {filename}")

Downloaded approximately 10% of yelp/yelp_academic_dataset_user.json


In [ ]:
# Read Yelp datasets into Spark DataFrames

base_dir = "file:///tmp/"

user_df = spark.read.json(base_dir + "yelp_academic_dataset_user.json")
business_df = spark.read.json(base_dir + "yelp_academic_dataset_business.json")
reviews_df = spark.read.json(base_dir + "yelp_academic_dataset_review.json")
checkin_df = spark.read.json(base_dir + "yelp_academic_dataset_checkin.json")
tip_df = spark.read.json(base_dir + "yelp_academic_dataset_tip.json")

In [ ]:
user_df.printSchema()
business_df.printSchema()
reviews_df.printSchema()
checkin_df.printSchema()
tip_df.printSchema()

In [ ]:
user_df.show(2)

In [ ]:
#show records vertically
business_df.show(2,vertical=True, truncate=False)

In [ ]:
checkin_df.show(2,truncate=False)

In [ ]:
tip_df.show(2,truncate=False)

### Partioning

Apache Spark can run a single concurrent task for every partition of an RDD, up to the total number of cores in the cluster.The number of partitions in spark should be decided thoughtfully based on the cluster configuration and requirements of the application. Having too large a number of partitions or too few - is not an ideal solution. 

In [ ]:
#display number of records by partition
def displaypartitions(df):
    #number of records by partition
    num = df.rdd.getNumPartitions()
    print("Partitions:", num)
    df.withColumn("partitionId", F.spark_partition_id())\
        .groupBy("partitionId")\
        .count()\
        .orderBy(F.asc("count"))\
        .show(num)

In [ ]:
#number of partitions
business_df.rdd.getNumPartitions()

In [ ]:
displaypartitions(business_df)

#### Repartition
Data can be repartitioned using the PySpark <code>repartition()</code> method randomly by increasing or decreasing the number of partitions, or repartitioned by selected columns.

In [ ]:
#re-partition the data
business_df = business_df.repartition(32)
displaypartitions(business_df)

#### Coalesce
When you want to reduce the number of partitions, it is recommended to use PySpark <code>coalesce()</code> over <code>repartition()</code> as it uses fewer resources due to less number of shuffles it takes.

In [ ]:
#coalesce the partitions
business_df = business_df.coalesce(10)
displaypartitions(business_df)

In [ ]:
business_df.select('postal_code').distinct().show()

In [ ]:
#repartition based on one column
business_df = business_df.repartition(F.col("postal_code"))
displaypartitions(business_df)

In [ ]:
#repartition based on one or more columns
business_df = business_df.repartition(F.col("postal_code"),F.col("city"))
displaypartitions(business_df)

### Exercise: Data Quality Checks
    
1. Peform basic data checks on business_df - count and stats
2. Count missing values in each column

### Exercise: Cleanup 

1) Drop duplicate business_id the business_df
2) Remove rows with categories that are empty/missing/null  

### Exercise: Group, count and plot 

Spark provides both APIs and SQL to explore and analyze the data.

1) List unique business categories
2) Group by and count distinct categories, sort in descending order
3) Create bar plot for the following:
    - a) category counts
    - b) city counts
    - c) state counts
    - d) star ratings 

### Exercise: Filtering

1) How many restaurants in IL ?
2) Find all reviews that include the word "terrible"

### Exercise: Adding Columns

For the business_df add a high level category business_type called Restaurant or Other

### Exercise: Date Operations

1) Write queries to tablate the Usefulness of review vs. Time since joined Yelp
2) Create function that calculates time difference (in days) and add a column called yelping_duration

### Exercise: Parititions and Query Performance 

What is the effect on performance for the query "SELECT date, stars, useful FROM reviews" for various partition sizes. Plot the results 

### Exercise: User Engagement

Plot the number of Users by Year of Yelp Membership

### Exercise: Correlations

Compute correlations between numeric columns and plot heatmap

### Flattening JSON columns

In [ ]:
business_df.select('attributes').show(5,truncate=False)

In [ ]:
# Get all attribute keys from the original DataFrame
attribute_keys = business_df.select("attributes.*").columns
print(attribute_keys)

# Create a new DataFrame with flattened attributes
business_attributes_df = business_df.select(
    'business_id',
    *[col(f"attributes.{key}").alias(key) for key in attribute_keys]
)

# Show the first few rows of the flattened DataFrame
business_attributes_df.show(5)

###  Exercise: Queries

#### 1) What are the counts for Alcohol serving types ?

#### 2) What percentages of business have Wifi ?

#### 3) Count and plot count of ambience types.